# Unit 2 code companion: A Map of Models

Fit the same data two ways and see exactly which questions each kind of model can answer, then compare two candidates honestly.

Run the cells in order. Each section matches a moment in the slides. The point is to see the mechanism move when you change the inputs, so change them.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

cars = pd.read_csv("https://richardson.byu.edu/220/cars.csv").dropna()
bikes = pd.read_csv("https://richardson.byu.edu/220/bikes.csv").dropna()
PREDS = ["weight", "horsepower", "model_year"]
X, y = cars[PREDS], cars["mpg"]
print(cars.shape, bikes.shape)

(392, 9) (365, 8)


## 1. Two kinds of model on the same data

One names a distribution for mpg. The other just predicts it.

In [2]:
lin = smf.ols("mpg ~ weight + horsepower + model_year", data=cars).fit()
forest = RandomForestRegressor(n_estimators=300, random_state=0).fit(X, y)

print("linear, in-sample R^2:", round(lin.rsquared, 3))
print("forest, in-sample R^2:", round(forest.score(X, y), 3))

linear, in-sample R^2: 0.808
forest, in-sample R^2: 0.981


The forest fits better. That is not the interesting part. What each one will *tell you* is.

## 2. What each one hands back



In [3]:
print(lin.summary().tables[1])
print("\nAIC:", round(lin.aic, 1))

print("\n--- the forest ---")
imp = pd.Series(forest.feature_importances_, index=PREDS).sort_values(ascending=False)
print(imp.round(3).to_string())

for attr in ["pvalues", "conf_int", "aic"]:
    print(f"forest.{attr}:", "yes" if hasattr(forest, attr) else "does not exist")

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -13.7194      4.182     -3.281      0.001     -21.941      -5.498
weight        -0.0064      0.000    -15.768      0.000      -0.007      -0.006
horsepower    -0.0050      0.009     -0.530      0.597      -0.024       0.014
model_year     0.7487      0.052     14.365      0.000       0.646       0.851

AIC: 2082.8

--- the forest ---
weight        0.672
horsepower    0.182
model_year    0.146
forest.pvalues: does not exist
forest.conf_int: does not exist
forest.aic: does not exist


Importances, and nothing else. No distribution means no likelihood, and no likelihood means there is no standard error, no $p$-value and no AIC to report. Those quantities are undefined here rather than hidden.

## 3. The type of y picks the probability model

Linear for a number, logistic for a yes/no, Poisson for a count.

In [4]:
# a number
m_num = smf.ols("mpg ~ weight", data=cars).fit()

# a yes/no
cars["efficient"] = (cars["mpg"] > 30).astype(int)
m_bin = smf.logit("efficient ~ weight", data=cars).fit(disp=0)

# a count
m_cnt = smf.glm("Count ~ Temperature", data=bikes, family=sm.families.Poisson()).fit()

print(f"linear   slope on weight: {m_num.params['weight']:.5f}")
print(f"logistic slope on weight: {m_bin.params['weight']:.5f}   (log-odds)")
print(f"Poisson  slope on temp  : {m_cnt.params['Temperature']:.5f}   (log-rate)")

print(f"\nlogistic predictions stay in [0,1]: {m_bin.predict().min():.3f} to {m_bin.predict().max():.3f}")
print(f"Poisson predictions stay positive : {m_cnt.predict().min():.1f} to {m_cnt.predict().max():.1f}")

linear   slope on weight: -0.00765
logistic slope on weight: -0.00373   (log-odds)
Poisson  slope on temp  : 0.03215   (log-rate)

logistic predictions stay in [0,1]: 0.000 to 0.877
Poisson predictions stay positive : 249.9 to 1295.6


## 4. What happens if you force the wrong one

An ordinary regression on a yes/no outcome runs. Look at what it returns.

In [5]:
bad = smf.ols("efficient ~ weight", data=cars).fit().predict()
print(f"predicted 'probabilities' range from {bad.min():.2f} to {bad.max():.2f}")
print("how many are not possible probabilities:", int(((bad < 0) | (bad > 1)).sum()))

predicted 'probabilities' range from -0.32 to 0.55
how many are not possible probabilities: 76


## 5. The type of y does not rule out a forest

The same algorithm has a regression mode and a classification mode.

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_num = RandomForestRegressor(n_estimators=100, random_state=0).fit(X, y)
rf_bin = RandomForestClassifier(n_estimators=100, random_state=0).fit(X, cars["efficient"])

print("numeric outcome ->", type(rf_num).__name__, "  predicts:", rf_num.predict(X[:3]).round(1))
print("yes/no outcome  ->", type(rf_bin).__name__, "predicts:",
      rf_bin.predict_proba(X[:3])[:, 1].round(2), "(probabilities)")

numeric outcome -> RandomForestRegressor   predicts: [17.6 14.8 17.4]
yes/no outcome  -> RandomForestClassifier predicts: [0. 0. 0.] (probabilities)


The outcome never rules a forest out. What rules it out is needing a coefficient.

## 6. Is model A better than model B?

Same rows, same measure, scored where neither model was fitted.

In [7]:
folds = KFold(5, shuffle=True, random_state=0)   # one fold object, used for every candidate

for name, mod in [("linear regression", LinearRegression()),
                  ("random forest", RandomForestRegressor(n_estimators=300, random_state=0))]:
    fit = mod.fit(X, y)
    train_mse = np.mean((fit.predict(X) - y) ** 2)
    cv = -cross_val_score(mod, X, y, cv=folds, scoring="neg_mean_squared_error")
    print(f"  {name:<20} training MSE {train_mse:6.2f}   CV MSE {cv.mean():6.2f}"
          f"   RMSE {np.sqrt(cv.mean()):5.2f} mpg")

  linear regression    training MSE  11.65   CV MSE  11.88   RMSE  3.45 mpg


  random forest        training MSE   1.18   CV MSE   8.02   RMSE  2.83 mpg


Read the two columns against each other. Training error says how well a model reproduces rows it already saw. The cross-validated column is the one that counts, and RMSE is the version you quote to a person, because it is back in the units of $y$.

## 7. The other way to compare: AIC

Needs a likelihood, so probability models only, and only on identical rows.

In [8]:
for f in ["mpg ~ weight",
          "mpg ~ weight + horsepower",
          "mpg ~ weight + horsepower + model_year"]:
    m = smf.ols(f, data=cars).fit()
    k = int(m.df_model) + 1
    print(f"  {f:<44} k={k}  logL={m.llf:8.1f}  AIC={m.aic:8.1f}")

full = smf.ols("mpg ~ weight + horsepower + model_year", data=cars).fit()
print(f"\ncheck the arithmetic: 2k - 2*logL = {2 * 4 - 2 * full.llf:.1f}")

  mpg ~ weight                                 k=2  logL= -1130.0  AIC=  2263.9
  mpg ~ weight + horsepower                    k=3  logL= -1121.0  AIC=  2248.0
  mpg ~ weight + horsepower + model_year       k=4  logL= -1037.4  AIC=  2082.8

check the arithmetic: 2k - 2*logL = 2082.8


Lower is better, and only differences mean anything. You could not put the forest on this table at all, which is why the cross-validated comparison above exists.

## What to take away




1. The type of `y` picks the probability model, and rules out nothing on the other side.
2. A model that names no distribution has no p-value, no confidence interval and no AIC.
3. Compare candidates on the same rows, with the same measure, where neither was fitted.
4. AIC needs a likelihood. Cross-validated error works on anything.
